# 27 — Prompt Portability and Multi-Model Systems

## Scenario
Your company has hardcoded hundreds of API calls to a single AI provider throughout your codebase. 

**The Problem:** The provider suffers a massive outage. Your entire application goes down because you are vendor-locked. Furthermore, migrating to a new model requires rewriting business logic.

**The Solution:** We must decouple the application from the model. We do this by defining a strict **Contract** (a Pydantic schema) and using **Model Adapters** to translate the contract into provider-specific prompts. We then use a **Router** to implement automatic fallbacks.

In [ ]:
from typing import List, Optional
from pydantic import BaseModel, Field
from google import genai
from google.genai import types

client = genai.Client()
PRIMARY_MODEL = 'gemini-2.5-flash'


## Step 1: The Universal Contract

This Pydantic schema is the ONLY thing the core application knows about. It knows nothing about "system instructions" or "XML tags."

In [ ]:
class FinancialSummary(BaseModel):
    company_name: str
    revenue_trend: str = Field(description="UP, DOWN, or FLAT")
    key_risks: List[str] = Field(description="List of risk factors mentioned")


## Step 2: The Model Adapters

Adapters take the raw input, apply model-specific prompting techniques, call the API, and return the Pydantic contract.

In [ ]:
class GeminiAdapter:
    def generate_summary(self, text: str, force_failure: bool = False) -> FinancialSummary:
        if force_failure:
            raise Exception("503 Service Unavailable: Primary provider is down!")
            
        print("[GeminiAdapter] Calling Gemini API...")
        # Gemini excels with clear Markdown block instructions
        prompt = f"Extract financial data from the text.\n\nText:\n{text}"
        
        response = client.models.generate_content(
            model=PRIMARY_MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=0.0,
                response_mime_type="application/json",
                response_schema=FinancialSummary,
            )
        )
        return FinancialSummary.model_validate_json(response.text)

class FallbackAdapter:
    def generate_summary(self, text: str) -> FinancialSummary:
        print("[FallbackAdapter] Calling secondary provider (Simulated)...")
        # A different provider might require XML tags instead of Markdown
        prompt = f"<instruction>Extract financial data</instruction>\n<text>{text}</text>"
        
        # Simulating a successful response from a secondary provider
        return FinancialSummary(
            company_name="Acme Corp (From Fallback)", 
            revenue_trend="DOWN",
            key_risks=["Supply chain issues"]
        )


## Step 3: The Multi-Model Router

The Router ties it together, trying the primary and falling back to the secondary on failure.

In [ ]:
class ModelRouter:
    def __init__(self):
        self.primary = GeminiAdapter()
        self.fallback = FallbackAdapter()
        
    def get_summary(self, text: str, simulate_outage: bool = False) -> FinancialSummary:
        try:
            return self.primary.generate_summary(text, force_failure=simulate_outage)
        except Exception as e:
            print(f"\n🚨 [ROUTER ALERT] Primary model failed: {e}")
            print("🔄 [ROUTER] Failing over to secondary model...\n")
            return self.fallback.generate_summary(text)


## Step 4: Testing the Portability

We run the router normally, and then we simulate a massive cloud outage.

In [ ]:
router = ModelRouter()
document = "Acme Corp saw a 20% drop in revenue this quarter due to supply chain disruptions in Asia."

print("=== TEST 1: Normal Operations ===")
result_1 = router.get_summary(document)
print(result_1)

print("\n=== TEST 2: Primary Provider Outage ===")
result_2 = router.get_summary(document, simulate_outage=True)
print(result_2)


## Conclusion

Because the application logic only expects a `FinancialSummary` object, it **doesn't care** which model actually fulfilled the request.

By using **Pydantic Contracts** and **Model Adapters**, we achieved true Prompt Portability, allowing us to survive a vendor outage with zero application downtime.